# 1 — Download the raw corpora

The paper uses three corpora. Only the first can be downloaded by script; the
other two are served through web interfaces that have no bulk endpoint, so this
notebook automates what can be automated and gives the exact manual procedure
for the rest. **`docs/DATA.md` is the authoritative description** of where each
corpus comes from, which version is used and under which licence — read it
alongside this notebook.

| corpus | used for | size | how |
| --- | --- | --- | --- |
| **SPGC** (Standardized Project Gutenberg Corpus) | everything except Figure S4 and Table S9 | ~8 GB | scripted |
| **COREFL** (Corpus of English as a Foreign Language) | Figure S4, learners vs natives | ~4 MB | manual download, scripted import |
| **PARSEME 1.1** | Table S9, annotated multi-word expressions | ~95 MB | manual download |

Nothing downloaded here is redistributed by this repository. After notebook 2
has built the reduced data, everything in `data_raw/` can be deleted: no figure
reads it again.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 1.1 SPGC — scripted

`src/download_bulk.py` fetches three files from Zenodo record 2422561 into
`data_raw/`: the metadata CSV (~10 MB), the 1-gram counts (~1.5 GB) and the
ordered token streams (~6.4 GB). The download is resumable (`curl -L -C -`),
idempotent — it skips files already present and valid — and integrity-checked
with `zipfile.testzip`.

Run the cell below once. On a first download it is bound by the connection —
8 GB — and re-running it after an interruption resumes rather than restarting.

**With the files already in place it still costs about two minutes**, and does
not print anything while it works: before it can decide to skip a file it checks
the size against Zenodo and runs `zipfile.testzip` over the whole archive, which
decompresses all 8 GB. That is the integrity check paying for itself, not a
stall.

In [ ]:
py("download_bulk.py")

Verify without downloading anything. This is also the cell to run if you already
have the SPGC bulk somewhere and copied or symlinked it into `data_raw/`.

It repeats the same `testzip` pass as the cell above, so on an already-complete
copy the two cells together take roughly **four minutes** — the measured figure
is 3 min 43 s. Skip this one if you have just run the previous cell; it is here
for the case where the bulk arrived by some other route.

It also compares the MD5 of each file with `manifests/spgc_checksums.json`.
Size and `testzip` only say that a file is a complete zip; the MD5 is what says
it is *this* release of SPGC. If it does not match you do not have the corpus
the paper was computed on, whatever the file names say — so this cell reports
rather than stopping the notebook, and you should read what it prints.

In [ ]:
py("download_bulk.py", "--check", must_succeed=False)

## 1.2 COREFL — manual, and why

COREFL is served by a web interface only: there is no bulk-download URL and no
API (`/search` and `/search_simple` return the same JavaScript shell, with no
form action and no archive endpoint), so this one fetch cannot be scripted.
Everything after it is.

Repeat these steps **once per subcorpus**:

1. open <http://corefl.learnercorpora.com/search_simple>
2. **Subcorpus** → `Learners of L2 English`; **L1** → `L1 any`
3. leave **Words (optional)** *empty* — an empty query returns the whole
   subcorpus — and press **Search**
4. press **Download** at the bottom right of the result list
5. in the dialog: purpose → *For research*; **format → `Texts only`**
6. repeat from step 2 with **Subcorpus** set to the native-speaker controls

Choose **`Texts only`**. `Texts with metadata` prepends a header to each file
that the tokeniser would count as running text, and the CSV exports need a
different reader. Nothing is lost: the file names carry the metadata the
pipeline uses.

Then extract every archive into one folder — the importer classifies by file
name, so the layout does not matter — and check the file count against the
"Results 1 to 50 of N" line the site showed:

```bash
mkdir -p ~/Downloads/corefl && cd ~/Downloads/corefl
unzip ~/Downloads/<learners>.zip && unzip ~/Downloads/<natives>.zip
find ~/Downloads/corefl -name '*.txt' | wc -l
```

### Import, and pin exactly which copy this is

COREFL is versioned, and v2.0 added L1s that v1 did not have (L1 Chinese, for
instance), so "download COREFL" alone does not identify the data. `--write-manifest`
records a sha256 per file plus a free-text note in `manifests/corefl_manifest.json`;
`--check` re-verifies any later copy against it. Anyone re-running this pipeline
gets identical numbers only if `--check` passes.

Two steps, in this order. `--from-dir` **imports**: it reads the extracted
download and copies each file into `data_raw/corefl/<group>/`, classifying it by
its *name* rather than by the folder it arrived in. `--check` then **verifies**
that imported copy against `manifests/corefl_manifest.json`, which records a
sha256 per file plus a note describing the copy the paper used.

`--check` alone is not enough on a fresh clone: there is nothing to check until
the import has run.

In [ ]:
# 1. import: classify the extracted files by name into data_raw/corefl/<group>/
#    point COREFL_DOWNLOAD at wherever you extracted the two archives
COREFL_DOWNLOAD = os.path.join(REPO, "data_raw", "corefl_download")
print("importing from:", COREFL_DOWNLOAD)
py("import_corefl.py", "--from-dir", COREFL_DOWNLOAD)

In [ ]:
# 2. verify the imported copy against the committed manifest.
#    This must pass, or your COREFL copy is not the one the paper used. It is
#    one of the two steps here that report rather than stop, so that a clone
#    which only wants SPGC can still finish this notebook -- read the banner.
py("import_corefl.py", "--check", must_succeed=False)

In [ ]:
# Re-pin to a DIFFERENT copy -- only if you deliberately mean to replace the
# reference. This rewrites manifests/corefl_manifest.json and any later --check
# then verifies against your copy rather than the paper's.
# py("import_corefl.py", "--from-dir", COREFL_DOWNLOAD, "--write-manifest",
#    "--note", "v2.0 Oct 2025, subcorpora: learners + natives, Texts only, "
#              "downloaded 2026-07-26")

## 1.3 PARSEME 1.1 — manual

PARSEME is the only manually annotated multi-word-expression resource covering
all five languages of the paper. It is distributed as per-language folders of
`.cupt` files (CoNLL-U Plus), which the pipeline reads directly.

Download it from the LINDAT/CLARIN repository and extract the five languages
into `data_raw/parseme_1_1/`, one folder per language, each containing
`train.cupt`, `dev.cupt` and `test.cupt`. The exact record, version and licence
are in `docs/DATA.md`.

The cell below only checks that the layout is the one `src/mwe_ranks.py` expects.

In [ ]:
import os

PARSEME = os.path.join(REPO, "data_raw", "parseme_1_1")
EXPECTED = ("EN", "FR", "IT", "ES", "DE")

if not os.path.isdir(PARSEME):
    print(f"not found: {PARSEME}\nsee docs/DATA.md -- Table S9 cannot be rebuilt without it")
else:
    for folder in EXPECTED:
        path = os.path.join(PARSEME, folder)
        splits = [s for s in ("train.cupt", "dev.cupt", "test.cupt")
                  if os.path.exists(os.path.join(path, s))]
        print(f"{folder}: {'ok' if splits else 'MISSING'}  {splits}")

## What you have now

`data_raw/` holds the three corpora. Nothing in it is committed, and nothing
downstream reads it once notebook 2 has run — so it can be deleted afterwards to
recover the disk.

**Next:** `02_build_reduced_data.ipynb`.